# <center> Clase 11: NLP mediante LLMs</center>



**Agenda:**

    11.1.- Técnicas de customización
      11.1.1.- Prompt engineering y OutputParsing
      11.1.2.- Técnicas de inclusión de material auxiliar: RAG
      11.1.3.- Ajuste mediante adaptadores: PEFT
      11.1.4.- Fine-tuning mediante active learning: RLHF and DPO

    11.2.- Software y herramientas
      11.2.1.- Librerías Python: LangChain, Llama-index y Hugging Face
      11.2.2.- Vector Databases
      11.2.3.- Plataformas cloud

    11.3.- Otras técnicas avanzadas
      11.3.1.- Creación de agentes mediate ReAct
      11.3.2.- Tecnologías auxiliares: Ollana, MoE, LangSmith, Ray, etc.

     

## 11.1.- Técnicas de customización

### 11.1.1.- Prompt engineering y OutputParsing

En el prompt engineering, la descripción de la tarea que debe realizar la IA está incrustada en la entrada como una pregunta, sugerencia o formato explícito de salida (output-parsing), por ejemplo, consumiendo con ello una parte de la ventana de contexto. La ingeniería rápida generalmente funciona convirtiendo una o más tareas en un conjunto de datos basado en mensajes y entrenando un modelo de lenguaje con lo que se ha llamado "aprendizaje basado en mensajes" o simplemente "aprendizaje rápido". La amplia accesibilidad de estas herramientas fue impulsada por la publicación de varios cuadernos de código abierto y proyectos dirigidos por la comunidad para la síntesis de imágenes.

In [ ]:
# from https://colab.research.google.com/github/sugarforever/LangChain-Tutorials/blob/main/LangChain_Output_Parsing.ipynb

### 11.1.2.- Técnicas de inclusión de material auxiliar: RAG

EL RAG (Retrieval Augmented Generation) combina un componente de recuperación de información con un modelo de generador de texto. RAG se puede ajustar y su conocimiento interno se puede modificar de manera eficiente y sin necesidad de volver a entrenar todo el modelo.

In [2]:
# from https://colab.research.google.com/github/deepset-ai/haystack-tutorials/blob/main/tutorials/07_RAG_Generator.ipynb

### 11.1.3.- Ajuste mediante adaptadores: PEFT

Los métodos de ajuste fino de parámetros eficientes (PEFT) permiten una adaptación eficiente de modelos de lenguaje previamente entrenados (PLM) a diversas aplicaciones posteriores sin necesidad de ajustar todos los parámetros del modelo. El ajuste fino de los PLM a gran escala suele ser prohibitivamente costoso. En este sentido, los métodos PEFT solo ajustan una pequeña cantidad de parámetros (adicionales) del modelo, lo que reduce en gran medida los costos computacionales y de almacenamiento. Las recientes técnicas PEFT de última generación logran un rendimiento comparable al del ajuste completo.

In [ ]:
# from https://colab.research.google.com/github/michael-franke/npNLG/blob/main/neural_pragmatic_nlg/07-LLMs/07b-pretrained-LLMs.ipynb

### 11.1.4.- Fine-tuning mediante active learning: RLHF and DPO

El Reinforcement Learnining from Human Feed-back (RLHF) es una técnica que entrena un "modelo de recompensa" directamente a partir de la retroalimentación humana y utiliza el modelo como una función de recompensa para optimizar la policy de un agente mediante aprendizaje por refuerzo, principalmente mediante PPO. El modelo de recompensa se entrena antes de que se optimice la política para predecir si un resultado determinado es bueno (recompensa alta) o malo (recompensa baja). La retroalimentación humana se recopila más comúnmente pidiéndoles que clasifiquen instancias del comportamiento del agente. Estas clasificaciones pueden utilizarse luego para calificar los resultados, por ejemplo con el sistema de calificación Elo. Si bien el juicio de preferencia se adopta ampliamente, existen otros tipos de retroalimentación humana que brindan información más rica, como la retroalimentación numérica, la retroalimentación en lenguaje natural y la tasa de edición.

In [6]:
# from https://medium.com/@csakash03/fine-tuning-llama-2-llm-on-google-colab-a-step-by-step-guide-cf7bb367e790

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q trl xformers wandb datasets einops gradio sentencepiece bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline, logging, TextStreamer
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
import os,torch, wandb, platform, gradio, warnings
from datasets import load_dataset
from trl import SFTTrainer
from huggingface_hub import notebook_login

In [ ]:
def print_system_specs():
    # Check if CUDA is available
    is_cuda_available = torch.cuda.is_available()
    print("CUDA Available:", is_cuda_available)
# Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print("Number of CUDA devices:", num_cuda_devices)
    if is_cuda_available:
        for i in range(num_cuda_devices):
            # Get CUDA device properties
            device = torch.device('cuda', i)
            print(f"--- CUDA Device {i} ---")
            print("Name:", torch.cuda.get_device_name(i))
            print("Compute Capability:", torch.cuda.get_device_capability(i))
            print("Total Memory:", torch.cuda.get_device_properties(i).total_memory, "bytes")
    # Get CPU information
    print("--- CPU Information ---")
    print("Processor:", platform.processor())
    print("System:", platform.system(), platform.release())
    print("Python Version:", platform.python_version())
print_system_specs()

In [ ]:
# Pre trained model
model_name = "meta-llama/Llama-2-7b-hf" 

# Dataset name
dataset_name = "vicgalle/alpaca-gpt4"

# Hugging face repository link to save fine-tuned model(Create new repository in huggingface,copy and paste here)
new_model = "Repository link here"

In [ ]:
# Log into hugging face hub
notebook_login()

In [ ]:
# Load dataset (you can process it here)
dataset = load_dataset(dataset_name, split="train[0:10000]")
dataset["text"][0]

In [ ]:
# Load base model(llama-2-7b-hf) and tokenizer
bnb_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_quant_type= "nf4",
    bnb_4bit_compute_dtype= torch.float16,
    bnb_4bit_use_double_quant= False,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False # silence the warnings. Please re-enable for inference!
model.config.pretraining_tp = 1
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_eos_token = True
tokenizer.add_bos_token, tokenizer.add_eos_token

In [ ]:
# Monitoring login
wandb.login(key="Enter the Authorization code here")
run = wandb.init(project='Fine tuning llama-2-7B', job_type="training", anonymous="allow")

In [ ]:
# LoRA config
peft_config = LoraConfig(
    lora_alpha= 8,
    lora_dropout= 0.1,
    r= 16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj"]
)

# Training arguments
training_arguments = TrainingArguments(
    output_dir= "./results",
    num_train_epochs= 1,
    per_device_train_batch_size= 8,
    gradient_accumulation_steps= 2,
    optim = "paged_adamw_8bit",
    save_steps= 1000,
    logging_steps= 30,
    learning_rate= 2e-4,
    weight_decay= 0.001,
    fp16= False,
    bf16= False,
    max_grad_norm= 0.3,
    max_steps= -1,
    warmup_ratio= 0.3,
    group_by_length= True,
    lr_scheduler_type= "linear",
    report_to="wandb",
)

# Setting sft parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length= None,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    packing= False,
) 

In [ ]:
# Train model
trainer.train()

In [ ]:
# Save the fine-tuned model
trainer.model.save_pretrained(new_model)
wandb.finish()
model.config.use_cache = True
model.eval()

In [ ]:
# Test the new model
def stream(user_prompt):
    runtimeFlag = "cuda:0"
    system_prompt = 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n'
    B_INST, E_INST = "### Instruction:\n", "### Response:\n"
    prompt = f"{system_prompt}{B_INST}{user_prompt.strip()}\n\n{E_INST}"
    inputs = tokenizer([prompt], return_tensors="pt").to(runtimeFlag)
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    # Despite returning the usual output, the streamer will also print the generated text to stdout.
    _ = model.generate(**inputs, streamer=streamer, max_new_tokens=500)

In [ ]:
stream("what is newtons 3rd law and its formula")

## 11.2.- Software y herramientas

### 11.2.1.- Librerías Python

#### LangChain

Una de estas librerías es LangChain. LangChain es un framwork de desarrollo de aplicaciones basadas en modelos de lenguaje. Permite aplicaciones que son conscientes del contexto, es decir, conectan un modelo de lenguaje con fuentes de contexto (instrucciones rápidas, algunos ejemplos, contenido en el que fundamentar su respuesta, etc.) y que además permiten razonar, es decir, confiar en un modelo de lenguaje para razonar (sobre cómo responder según el contexto proporcionado, qué acciones tomar, etc.). Este framework consta de varias extensiones:

- **LangChain Libraries**: integraciones para una gran variedad de componentes, un tiempo de ejecución básico para combinar estos componentes en cadenas y agentes.
- **LangChain Templates**: arquitecturas de referencia fácilmente implementables.
- **LangServe**: una biblioteca para implementar cadenas LangChain como API REST.
- **LangSmith**: una plataforma de desarrollo que le permite depurar, probar, evaluar y monitorear cadenas creadas en cualquier marco LLM y se integra perfectamente con LangChain.


In [ ]:
# from https://colab.research.google.com/drive/1VOwJpcZqOXag-ZXi-52ibOx6L5Pw-YJi?usp=sharing#scrollTo=nTDgRy0jKDkP

#### Llama-index

Llama-index, por su parte, en gran medida se solapa con las funcionalidades de LangChain y de Hugging Face, pero se recoimienda utilizarse en combinación con éstas, pues su parte fuerte es la integración con fuentes de datos y bases de datos vectoriales. 

In [ ]:
# from https://colab.research.google.com/github/PradipNichite/Youtube-Tutorials/blob/main/LlamaIndex_Tutorial.ipynb

### 11.2.2.- Vector databases

Las Vector Databases almacenan y brindan acceso a datos estructurados y no estructurados, como texto o imágenes, junto con sus embeddings. Los embeddings son la representación numérica de los datos como una larga lista de números que captura el significado semántico del objeto de datos original. Debido a que los objetos similares están muy juntos en el espacio vectorial, la similitud de los objetos de datos se puede calcular en función de la distancia entre las embeddings del objeto de datos. Esto abre la puerta a un nuevo tipo de técnica de búsqueda llamada búsqueda vectorial que recupera objetos en función de su similitud. A diferencia de la búsqueda tradicional basada en palabras clave, la búsqueda semántica ofrece una forma más flexible de buscar elementos.

Algunas de las tecnologías hoy en día más populares son:
- **Pinecone**: servicio web de gran fiabilidad y aceptación
- **Weaviate**: uno de los principales sistemas de código abierto
- **ChromaDB**: framework de especial sencillez y aplicabilidad.
- Otros similres: **Qdrant**, **Zilliz**, **Milvus**, **Jina**, etc.
- Bases de datos tradicionales adaptadas a LLMs: **Redis** y **ElasticSearch**.


In [5]:
# Pinecone example
# from https://colab.research.google.com/github/pinecone-io/examples/blob/master/integrations/openai/semantic_search_openai.ipynb
# from https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/llama-index/llama-index-intro.ipynb#scrollTo=YenjP38jtRdE

### 11.2.3.- Plataformas cloud

Los principales proveedores de plataformas web públicas, como Azure (vinculada con OpenAI), Amazon Web Services (AWS Bedrock) y Google (Vertex AI y Gen AI studio) ofrecen actualmente servicios web para una fácil explotación de la IA Generativa, especialmente modelos de lenguaje.

In [ ]:
# AWS Bedrock & ElasticSearch
# from https://colab.research.google.com/github/elastic/elasticsearch-labs/blob/main/notebooks/integrations/amazon-bedrock/langchain-qa-example.ipynb

## 11.3.- Otras técnicas avanzadas

### 11.3.1.- Creación de agentes mediate ReAct

In [ ]:
# from https://python.langchain.com/docs/modules/agents/agent_types/react

In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.llms import OpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import AIMessage, HumanMessage

In [ ]:
# Initialize tools
tools = [TavilySearchResults(max_results=1)]

#### Create agent

In [ ]:
# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/react")

# Choose the LLM to use
llm = OpenAI()

# Construct the ReAct agent
agent = create_react_agent(llm, tools, prompt)

# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor.invoke({"input": "what is LangChain?"})

#### Using chat history

In [1]:
# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/react-chat")

# Construct the ReAct agent
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke(
    {
        "input": "what's my name? Only use a tool if needed, otherwise respond with Final Answer",
        # Notice that chat_history is a string, since this prompt is aimed at LLMs, not chat models
        "chat_history": "Human: Hi! My name is Bob\nAI: Hello Bob! Nice to meet you",
    }
)

### 11.3.2.- Tecnologías auxiliares: Ollana, MoE, LangSmith, Ray, etc.

In [4]:
# from https://colab.research.google.com/github/langchain-ai/langsmith-cookbook/blob/main/tracing-examples/rest/rest.ipynb